In [ ]:
import anndata as ad
import scxmatch as xm
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
group = "D18"

In [ ]:
adata = ad.read_h5ad("../data/processed_schiebinger.hdf5")

In [ ]:
minority_class = np.where(adata.obs["perturbation"] == group)[0]
control_class = np.where(adata.obs["perturbation"] == "control")[0]

In [ ]:
len(minority_class)

In [ ]:
len(control_class)

In [ ]:
rows = list()
for k in [10, 100, 1000]:
    for subsample_factor_minority_class in [0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]:
        subsampled_minority_class_indices = np.random.choice(minority_class, size=int(subsample_factor_minority_class*len(minority_class)))
        subsampled_adata = adata[np.concatenate([subsampled_minority_class_indices, control_class])].copy()
        control_indices_subsampled = np.where(subsampled_adata.obs["perturbation"] == "control")[0]
        minority_indices_subsampled = np.where(subsampled_adata.obs["perturbation"] == group)[0]

        result = xm.test(subsampled_adata, k=k, group_by="perturbation", reference="control", test_group=group, metric="sqeuclidean")
    
        majority_coverage = len(np.where(subsampled_adata[control_indices_subsampled].obs[f'XMatch_partner_{group}_vs_control'].notna())[0])  / (len(control_indices_subsampled))
        minority_coverage = len(np.where(subsampled_adata[minority_indices_subsampled].obs[f'XMatch_partner_{group}_vs_control'].notna())[0])  / (len(minority_indices_subsampled))
        rows.append([subsample_factor_minority_class, k, minority_coverage, majority_coverage, result["p_value"], result["z_score"], result["coverage"]])

In [ ]:
results = pd.DataFrame(rows, columns=["Subsampling factor test group", "k", "Matching coverage in subsampled test group", "Matching coverage in control group", "P", "z", "s"])

In [ ]:
num_samples = results["Subsampling factor test group"] * len(minority_class)

In [ ]:
results["Ratio #samples test group: #samples control group"] = num_samples.astype(int).astype(str) + f"\n:{len(control_class)}"

In [ ]:
melted = pd.melt(results, id_vars=["Subsampling factor test group", "k", "P", "z", "s", "Ratio #samples test group: #samples control group"], var_name="Hue", value_name="Matching coverage")

In [ ]:
f, axs = plt.subplots(3, 2, sharex=True, sharey=False, figsize=(12, 5))
for i, (k, subdf) in enumerate(melted.groupby("k")):
    legend = i == 0
    sns.lineplot(subdf, y="Matching coverage", hue="Hue", ax=axs[i, 0], x="Ratio #samples test group: #samples control group", legend=False, ls="--", palette=sns.color_palette("viridis", 2))
    sns.scatterplot(subdf, y="Matching coverage", hue="Hue", ax=axs[i, 0], x="Ratio #samples test group: #samples control group", legend=legend, palette=sns.color_palette("viridis", 2))

    axs[i, 0].set_title(f"k={k}")
    plt.tight_layout()

for i, (k, subdf) in enumerate(results.groupby("k")):
    legend = i == 0
    sns.lineplot(subdf, y="P", ax=axs[i, 1], x="Ratio #samples test group: #samples control group", legend=False, ls="--", color=sns.color_palette("magma", 1)[0])
    sns.scatterplot(subdf, y="P", ax=axs[i, 1], x="Ratio #samples test group: #samples control group", legend=legend, color=sns.color_palette("magma", 1)[0])
    #axs[i, 1].axhline(np.log1p(0.001), c="gray", ls="dotted")
    axs[i, 1].set_title(f"k={k}")

plt.tight_layout()
    
sns.move_legend(axs[0, 0], bbox_to_anchor=(0.5,2.2), loc="upper center")
axs[0, 0].set_title("Coverage by group\nk=10")
axs[0, 1].set_title("$P$-value\nk=10") 
#plt.ylim(0, 1.1)
plt.savefig("../plots/supp/groupwise_coverage_D18.svg", bbox_inches="tight")

In [ ]:
results.to_csv(f"../evaluation_results/supp_groupwise_coverage_{group}.csv")

In [ ]:
results_D1 = pd.read_csv(f"../evaluation_results/supp_groupwise_coverage_D1.5.csv").drop("Unnamed: 0", axis=1)

In [ ]:
results_D18 = pd.read_csv(f"../evaluation_results/supp_groupwise_coverage_D18.csv").drop("Unnamed: 0", axis=1)

In [ ]:
melted_18 = pd.melt(results_D18, id_vars=["Subsampling factor test group", "k", "P", "z", "s", "Ratio #samples test group: #samples control group"], var_name="Hue", value_name="Matching coverage")
melted_1 = pd.melt(results_D1, id_vars=["Subsampling factor test group", "k", "P", "z", "s", "Ratio #samples test group: #samples control group"], var_name="Hue", value_name="Matching coverage")

In [ ]:
fig = plt.figure(figsize=(12, 12))

gs = fig.add_gridspec(
    7, 2,
    height_ratios=[1, 1, 1, 0.01, 1, 1, 1],  # spacer row in middle
    hspace=1.8
)

axs = np.empty((6, 2), dtype=object)

# Top block (rows 0,1,2)
for r in range(3):
    for c in range(2):
        axs[r, c] = fig.add_subplot(gs[r, c])

# Bottom block (rows 4,5,6 in GridSpec)
for r in range(3, 6):
    for c in range(2):
        axs[r, c] = fig.add_subplot(gs[r + 1, c])

# ------------------------------------------------------------------
# D18 panels (top 3 rows)
# ------------------------------------------------------------------
for i, (k, subdf) in enumerate(melted_18.groupby("k")):
    legend = i == 0

    sns.lineplot(
        subdf,
        y="Matching coverage",
        hue="Hue",
        ax=axs[i, 0],
        x="Ratio #samples test group: #samples control group",
        legend=False,
        ls="--",
        palette=sns.color_palette("viridis", 2),
    )

    sns.scatterplot(
        subdf,
        y="Matching coverage",
        hue="Hue",
        ax=axs[i, 0],
        x="Ratio #samples test group: #samples control group",
        legend=legend,
        palette=sns.color_palette("viridis", 2),
    )

    axs[i, 0].set_title(f"k={k}")

for i, (k, subdf) in enumerate(results_D18.groupby("k")):
    legend = i == 0

    sns.lineplot(
        subdf,
        y="P",
        ax=axs[i, 1],
        x="Ratio #samples test group: #samples control group",
        legend=False,
        ls="--",
        color=sns.color_palette("magma", 1)[0],
    )

    sns.scatterplot(
        subdf,
        y="P",
        ax=axs[i, 1],
        x="Ratio #samples test group: #samples control group",
        legend=legend,
        color=sns.color_palette("magma", 1)[0],
    )

    axs[i, 1].set_title(f"k={k}")

# ------------------------------------------------------------------
# D1.5 panels (bottom 3 rows)
# ------------------------------------------------------------------
for i, (k, subdf) in enumerate(melted_1.groupby("k")):

    sns.lineplot(
        subdf,
        y="Matching coverage",
        hue="Hue",
        ax=axs[i + 3, 0],
        x="Ratio #samples test group: #samples control group",
        legend=False,
        ls="--",
        palette=sns.color_palette("viridis", 2),
    )

    sns.scatterplot(
        subdf,
        y="Matching coverage",
        hue="Hue",
        ax=axs[i + 3, 0],
        x="Ratio #samples test group: #samples control group",
        legend=False,
        palette=sns.color_palette("viridis", 2),
    )

    axs[i + 3, 0].set_title(f"k={k}")

for i, (k, subdf) in enumerate(results_D1.groupby("k")):

    sns.lineplot(
        subdf,
        y="P",
        ax=axs[i + 3, 1],
        x="Ratio #samples test group: #samples control group",
        legend=False,
        ls="--",
        color=sns.color_palette("magma", 1)[0],
    )

    sns.scatterplot(
        subdf,
        y="P",
        ax=axs[i + 3, 1],
        x="Ratio #samples test group: #samples control group",
        legend=False,
        color=sns.color_palette("magma", 1)[0],
    )

    axs[i + 3, 1].set_title(f"k={k}")

# ------------------------------------------------------------------
# Legend
# ------------------------------------------------------------------
sns.move_legend(
    axs[0, 0],
    bbox_to_anchor=(0.5, 3),
    loc="upper center",
    ncol=1,
)

# ------------------------------------------------------------------
# Column titles for first row
# ------------------------------------------------------------------
axs[0, 0].set_title("Coverage by group\nk=10")
axs[0, 1].set_title("$P$-value\nk=10")

# ------------------------------------------------------------------
# Axis formatting
# ------------------------------------------------------------------
for i in range(6):
    axs[i, 0].set_ylim(0.95, 1.01)
    axs[i, 1].set_ylim(-0.01, 0.05)
    axs[i, 1].axhline(0.01, ls="dotted", c="gray")

fig.text(
    0.5,
    0.91,
    "Test group D18",
    ha="center",
    va="top",
    fontsize=12,
    fontweight="bold",
)

fig.text(
    0.5,
    0.44,
    "Test group D1.5",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
)

plt.savefig(
    "../plots/supp/groupwise_coverage.pdf",
    bbox_inches="tight",
)